# Chapter 23: Visualization

Source span used: *Fundamentals of Computer Graphics*, Chapter 23, printed pages 645-680 / PDF pages 662-697. The last page in the assigned PDF span is a blank separator before references, so the chapter content extracted here runs through the FAQ page immediately before the references.

## Chapter Goal

Visualization is the graphics discipline for problems where a person still has to judge, compare, explore, or explain. This notebook turns the chapter into a computational design workflow: start with data and task abstraction, choose marks and channels that fit the data type, add interaction and composition when a single view is insufficient, reduce data when pixels and attention run out, and validate both the visual design and the computation.

The running example is a synthetic monitoring dataset for a small building and compute facility. Each item is a sensor observation with spatial coordinates, categorical subsystem, ordered status, quantitative measurements, uncertainty, and a simulated anomaly outcome. The data are artificial so every artifact can be generated and checked without depending on external files.

## Translation guide

| Chapter idea | Computational translation in this notebook | What to inspect |
| --- | --- | --- |
| Domain problem | A user wants to triage sensor faults, explain load peaks, and report confidence. | Does every visual choice answer a task rather than merely decorate the page? |
| Data abstraction | A `pandas` table with quantitative, ordered, categorical, relational, temporal, and spatial fields. | Which fields are raw and which are derived? |
| Task abstraction | Operations such as find outliers, compare groups, follow graph paths, filter ranges, and summarize distributions. | Can the same task be stated without domain vocabulary? |
| Marks and visual channels | Points, lines, areas, cells, and glyphs encoded with position, color, size, lightness, and opacity. | Are precise quantities placed on spatial position before weaker channels? |
| Interaction | Selection predicates, hover details, legend filtering, zooming, and linked summaries. | What changes when a user filters or asks for detail? |
| Composition | Layering, small multiples, linked adjacent views, and node-link/matrix graph encodings. | Does comparison happen in the same frame or across coordinated frames? |
| Reduction | Aggregation, filtering, focus+context distortion, and PCA. | What information is deliberately removed, summarized, or emphasized? |
| Validation and uncertainty | Layered validation checks, artifact integrity checks, calibration, coverage, and invariant JSON files. | Does the notebook test the design claims with data, not just prose? |

## Visualization-planner pass: Visual storyboard

| Storyboard item | Artifact | Library route | Learner inspection target | Validation or invariant |
| --- | --- | --- | --- | --- |
| Data/task abstraction map | `figures/data-task-abstraction-map.png` | NetworkX plus Matplotlib for a dependency diagram | Follow the path from domain question to data operation to visual and check. | Directed acyclic graph, every task reaches a validation node. |
| Marks and channels ladder | `figures/marks-channels-encoding-ladder.png` | Matplotlib for durable static comparison | Compare point, line, area, and glyph encodings on the same synthetic data. | Category count stays within a hue-safe range; glyph values are normalized. |
| Color/channel validation | `figures/color-channel-fit-validation.png` | Matplotlib colormap analysis plus SciPy rank correlation | See why monotone lightness is safer for ordered/quantitative fields than rainbow hue order. | Sequential map has monotone luminance; rainbow has luminance reversals. |
| Overview, filter, details | `html/overview-filter-details-dashboard.html` | Plotly for zoom, hover, legends, and standalone HTML | Inspect high-risk selected items and their details without leaving the overview. | Selection predicate returns a nonempty strict subset. |
| Composition and linked views | `figures/composition-linked-views.png` | Matplotlib for layered and adjacent static views | Compare single drawing, layered focus/background, small multiples, and linked summaries. | Shared axes and consistent selected set across views. |
| Graph matrix comparison | `figures/graph-matrix-composition-comparison.png` | NetworkX for relational data, Matplotlib for matrix/node-link views | Compare path-following in node-link form against crossing-free adjacency matrix form. | Matrix edge count equals graph edge count. |
| Reduction workflow | `figures/reduction-aggregation-pca-workflow.png` | NumPy/SciPy/Pandas plus Matplotlib | Compare raw clutter, binned aggregation, filtered anomalies, and PCA projection. | Aggregation conserves item count; PCA axes are orthonormal. |
| Focus+context lens | `figures/focus-context-fisheye-lens.png` | NumPy radial transform plus Matplotlib | Watch a focus neighborhood expand while distant context remains present. | Radial order is monotone and the boundary remains fixed. |
| Validation and uncertainty | `figures/uncertainty-validation-workbench.png` | Matplotlib plus statistical summaries | Distinguish high predicted risk from high uncertainty and check calibration. | Calibration error and interval coverage meet explicit tolerances. |

The visual choices are deliberately mixed: Matplotlib gives stable labeled static artifacts for comparisons and checks; Plotly is used where interaction is the lesson; NetworkX is used where the data abstraction is relational; NumPy/SciPy/Pandas handle reduction and statistical validation.


In [ ]:
from pathlib import Path
import json
import math
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Circle
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the Fundamentals of Computer Graphics book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, display_artifact, save_json, save_matplotlib, save_plotly_html, save_table_csv
from utils.notebook_checks import assert_nonblank_image
from utils.plotting import PALETTE, close, style_axis

TOPIC = "chapter-23"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / TOPIC
FIG_DIR = ARTIFACT_ROOT / "figures"
HTML_DIR = ARTIFACT_ROOT / "html"
CHECK_DIR = ARTIFACT_ROOT / "checks"
TABLE_DIR = ARTIFACT_ROOT / "tables"
DATA_DIR = ARTIFACT_ROOT / "data"
for directory in [FIG_DIR, HTML_DIR, CHECK_DIR, TABLE_DIR, DATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(23)
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.titlesize": 11,
    "axes.labelsize": 9,
    "font.size": 9,
})

chapter_paths = {"figures": [], "html": [], "checks": [], "tables": []}


## 1. Data And Task Abstraction

A visualization begins before drawing. The first design decision is to separate domain language from data and task abstractions. In this synthetic example, a facilities team might say, "show me which areas need attention before the morning load spike." Computationally, that becomes a table of observations, a spatial coordinate system, a time variable, a categorical subsystem field, ordered status labels, quantitative sensor values, and derived variables such as predicted risk.

The notebook keeps both levels visible. The prose names the domain intent; the code exposes the abstract operations that a visualization can support: filter, compare, find outliers, summarize uncertainty, and follow relationships.


In [ ]:
groups = np.array(["Cooling", "Compute", "Office", "Transit"])
centers = np.array([[-2.4, 1.3], [1.9, 1.1], [-1.4, -1.5], [2.0, -1.4]])
prob = np.array([0.27, 0.35, 0.23, 0.15])
n_items = 760
g_idx = rng.choice(len(groups), size=n_items, p=prob)
xy = centers[g_idx] + rng.normal(scale=[0.55, 0.42], size=(n_items, 2))
hour = rng.integers(0, 24, size=n_items)
workday = ((hour >= 8) & (hour <= 18)).astype(float)
night = ((hour <= 5) | (hour >= 22)).astype(float)

group_load_base = np.array([280, 520, 190, 145])
group_temp_base = np.array([22.0, 29.5, 23.5, 21.0])
group_latency_base = np.array([25, 42, 18, 30])
load_kw = group_load_base[g_idx] + 95 * workday + 35 * np.sin((hour - 6) / 24 * 2 * np.pi) + rng.normal(0, 42, n_items)
temp_c = group_temp_base[g_idx] + 0.012 * load_kw + 1.8 * np.sin((hour - 14) / 24 * 2 * np.pi) + rng.normal(0, 1.3, n_items)
vibration = np.clip(0.35 + 0.23 * (g_idx == 0) + 0.18 * night + rng.gamma(1.8, 0.16, n_items), 0, None)
latency_ms = group_latency_base[g_idx] + 0.055 * load_kw + rng.normal(0, 8, n_items)
uncertainty = np.clip(0.035 + 0.035 * (np.abs(xy[:, 0]) > 2.1) + 0.025 * night + rng.uniform(0.0, 0.045, n_items), 0.025, 0.16)

logit = (
    -4.2
    + 0.020 * (temp_c - 24)
    + 0.0042 * (load_kw - 260)
    + 1.25 * (vibration > 0.92)
    + 0.52 * (g_idx == 1)
    + 0.44 * workday
    + 0.32 * (latency_ms > np.quantile(latency_ms, 0.82))
)
true_risk = 1 / (1 + np.exp(-logit))
risk_score = np.clip(true_risk + rng.normal(0, uncertainty * 0.42), 0.005, 0.995)
anomaly = rng.random(n_items) < true_risk
status = pd.cut(risk_score, bins=[0, 0.14, 0.32, 1.0], labels=["normal", "watch", "intervene"], include_lowest=True, ordered=True)

sensor = pd.DataFrame({
    "sensor_id": [f"S{i:04d}" for i in range(n_items)],
    "x": xy[:, 0],
    "y": xy[:, 1],
    "subsystem": groups[g_idx],
    "hour": hour,
    "load_kw": load_kw,
    "temperature_c": temp_c,
    "vibration": vibration,
    "latency_ms": latency_ms,
    "uncertainty": uncertainty,
    "true_risk": true_risk,
    "risk_score": risk_score,
    "status": status,
    "anomaly": anomaly,
})

hours = np.arange(24)
time_rows = []
for group, base, temp_base in zip(groups, group_load_base, group_temp_base):
    for h in hours:
        day_curve = base + 100 * (8 <= h <= 18) + 42 * np.sin((h - 6) / 24 * 2 * np.pi)
        if group == "Compute":
            day_curve += 55 * np.exp(-0.5 * ((h - 14) / 3.0) ** 2)
        if group == "Cooling":
            day_curve += 35 * np.exp(-0.5 * ((h - 16) / 4.0) ** 2)
        time_rows.append({"subsystem": group, "hour": int(h), "mean_load_kw": float(day_curve), "mean_temp_c": float(temp_base + 0.012 * day_curve)})
time_series = pd.DataFrame(time_rows)

status_order = {"normal": 0, "watch": 1, "intervene": 2}
group_palette = {"Cooling": PALETTE["teal"], "Compute": PALETTE["blue"], "Office": PALETTE["green"], "Transit": PALETTE["gold"]}
selected = (sensor["risk_score"] >= 0.36) & (sensor["uncertainty"] <= sensor["uncertainty"].quantile(0.72))
high_uncertainty = sensor["uncertainty"] >= sensor["uncertainty"].quantile(0.85)

data_dictionary = pd.DataFrame([
    {"field": "sensor_id", "type": "categorical item key", "role": "identity"},
    {"field": "x, y", "type": "spatial quantitative", "role": "given floor-plan position"},
    {"field": "subsystem", "type": "categorical", "role": "grouping and hue"},
    {"field": "hour", "type": "ordered cyclic", "role": "temporal comparison"},
    {"field": "load_kw, temperature_c, vibration, latency_ms", "type": "quantitative", "role": "raw measurements"},
    {"field": "risk_score", "type": "derived quantitative", "role": "triage score"},
    {"field": "status", "type": "derived ordered", "role": "coarse action state"},
    {"field": "uncertainty", "type": "derived quantitative", "role": "confidence and validation"},
    {"field": "anomaly", "type": "simulated categorical outcome", "role": "validation label"},
])

display(data_dictionary)
display(sensor.head(6))
print(f"Synthetic observations: {len(sensor)}; selected high-risk subset: {int(selected.sum())}")


In [ ]:
# A chapter-specific dependency graph: domain questions become abstract tasks,
# which constrain encodings and validation.
nodes = {
    "triage": (0.0, 2.0, "Domain\ntriage faults"),
    "explain": (0.0, 1.0, "Domain\nexplain peaks"),
    "report": (0.0, 0.0, "Domain\nreport confidence"),
    "table": (1.8, 2.2, "Data\ntable fields"),
    "space": (1.8, 1.4, "Data\nspatial positions"),
    "time": (1.8, 0.6, "Data\ntime series"),
    "relation": (1.8, -0.2, "Data\nrelations"),
    "find": (3.7, 2.1, "Task\nfind outliers"),
    "compare": (3.7, 1.25, "Task\ncompare groups"),
    "filter": (3.7, 0.4, "Task\nfilter ranges"),
    "path": (3.7, -0.45, "Task\nfollow paths"),
    "marks": (5.7, 2.0, "Encoding\nmarks/channels"),
    "views": (5.7, 1.0, "Interaction\nlinked views"),
    "reduce": (5.7, 0.0, "Reduction\naggregate/PCA"),
    "valid": (7.6, 1.45, "Validate\nperceptual fit"),
    "compute": (7.6, 0.55, "Validate\ncounts/residuals"),
    "uncertain": (7.6, -0.35, "Validate\nuncertainty"),
}
edges = [
    ("triage", "table"), ("triage", "space"), ("explain", "time"), ("report", "table"),
    ("table", "find"), ("space", "find"), ("table", "compare"), ("time", "compare"),
    ("table", "filter"), ("relation", "path"), ("find", "marks"), ("compare", "marks"),
    ("filter", "views"), ("path", "views"), ("find", "reduce"), ("compare", "reduce"),
    ("marks", "valid"), ("views", "valid"), ("reduce", "compute"), ("marks", "uncertain"),
    ("views", "uncertain"), ("reduce", "uncertain"),
]
G = nx.DiGraph(edges)
G.add_nodes_from(nodes)
assert nx.is_directed_acyclic_graph(G)

fig, ax = plt.subplots(figsize=(11, 4.6))
ax.set_axis_off()
layer_colors = {
    "Domain": "#eef2ff",
    "Data": "#e9fbf6",
    "Task": "#fff7df",
    "Encoding": "#edf7ff",
    "Interaction": "#edf7ff",
    "Reduction": "#f2f7e8",
    "Validate": "#fff0f0",
}
for u, v in edges:
    x1, y1, _ = nodes[u]
    x2, y2, _ = nodes[v]
    arrow = FancyArrowPatch((x1 + 0.47, y1), (x2 - 0.47, y2), arrowstyle="-|>", mutation_scale=10, lw=1.1, color="#6b7280", alpha=0.85)
    ax.add_patch(arrow)
for key, (x, y, label) in nodes.items():
    prefix = label.split("\\n", 1)[0]
    color = layer_colors.get(prefix, "#f8fafc")
    box = FancyBboxPatch((x - 0.47, y - 0.28), 0.94, 0.56, boxstyle="round,pad=0.04,rounding_size=0.06", fc=color, ec="#4b5563", lw=1.0)
    ax.add_patch(box)
    ax.text(x, y, label, ha="center", va="center", fontsize=8.2, color=PALETTE["ink"])
for x, title in [(0.0, "domain"), (1.8, "data"), (3.7, "task"), (5.7, "technique"), (7.6, "validation")]:
    ax.text(x, 2.68, title, ha="center", va="center", fontsize=10, fontweight="bold", color=PALETTE["ink"])
ax.set_xlim(-0.75, 8.35)
ax.set_ylim(-0.9, 2.95)
ax.set_title("Data/task abstraction: each visual decision inherits a task and a validation target", loc="left")

data_task_path = save_matplotlib(fig, TOPIC, "data-task-abstraction-map.png")
close(fig)

reachable_validation = {}
validation_nodes = {"valid", "compute", "uncertain"}
for task in ["find", "compare", "filter", "path"]:
    descendants = nx.descendants(G, task)
    reachable_validation[task] = sorted(descendants & validation_nodes)

data_task_checks = {
    "node_count": G.number_of_nodes(),
    "edge_count": G.number_of_edges(),
    "is_dag": nx.is_directed_acyclic_graph(G),
    "tasks_with_validation_paths": reachable_validation,
    "all_tasks_reach_validation": all(bool(v) for v in reachable_validation.values()),
}
data_task_checks_path = save_json(data_task_checks, TOPIC, "data-task-abstraction-checks.json")
chapter_paths["figures"].append(data_task_path)
chapter_paths["checks"].append(data_task_checks_path)

display_artifact(data_task_path, width=900)
display_artifact(data_task_checks_path)


## 2. Marks, Channels, And Data-Type Fit

The chapter's central encoding vocabulary is compact: marks carry values through channels. A point mark can encode two quantities precisely with horizontal and vertical position, then add lower-capacity channels such as color, size, or opacity. A line mark is useful when continuity or ordering matters. Area and cell marks trade precise item reading for dense summaries. Glyphs can expose several attributes at once, but the price is display area and unequal perceptual salience among the glyph parts.

The next artifact uses the same synthetic dataset in four different mark families. The goal is not to choose a single winner; it is to make the tradeoff visible. The high-risk points are easy to find with position plus size in the point view, temporal comparison is easier in the line view, density and mean risk are easier in the area/cell view, and glyphs expose multiple measurements only for a small selected subset.


In [ ]:
from matplotlib.patches import Rectangle

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
ax = axes[0, 0]
for group in groups:
    rows = sensor[sensor["subsystem"] == group]
    ax.scatter(rows["x"], rows["y"], s=18 + 165 * rows["risk_score"], c=group_palette[group], label=group, alpha=0.62, edgecolors="white", linewidths=0.35)
ax.scatter(sensor.loc[selected, "x"], sensor.loc[selected, "y"], s=38 + 220 * sensor.loc[selected, "risk_score"], facecolors="none", edgecolors=PALETTE["red"], linewidths=1.2, label="selected risk")
style_axis(ax, "Point marks: position + hue + size", equal=True, xlabel="floor x", ylabel="floor y")
ax.legend(loc="upper right", fontsize=7, frameon=True)

ax = axes[0, 1]
for group in groups:
    rows = time_series[time_series["subsystem"] == group]
    ax.plot(rows["hour"], rows["mean_load_kw"], color=group_palette[group], lw=2.0, label=group)
ax.fill_between([8, 18], [0, 0], [760, 760], color="#e5e7eb", alpha=0.35, label="workday window")
style_axis(ax, "Line marks: ordered time comparison", xlabel="hour", ylabel="mean load (kW)")
ax.set_xlim(0, 23)
ax.legend(loc="upper left", fontsize=7, ncol=2)

ax = axes[1, 0]
counts, xedges, yedges = np.histogram2d(sensor["x"], sensor["y"], bins=25)
risk_sum, _, _ = np.histogram2d(sensor["x"], sensor["y"], bins=[xedges, yedges], weights=sensor["risk_score"])
mean_risk_grid = np.divide(risk_sum, counts, out=np.full_like(risk_sum, np.nan, dtype=float), where=counts > 0)
im = ax.imshow(mean_risk_grid.T, origin="lower", extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]], cmap="cividis", vmin=0, vmax=max(0.55, float(np.nanmax(mean_risk_grid))))
style_axis(ax, "Area/cell marks: aggregate mean risk", equal=True, xlabel="floor x", ylabel="floor y")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03, label="mean risk")

ax = axes[1, 1]
feature_cols = ["load_kw", "temperature_c", "vibration", "latency_ms"]
feature_names = ["load", "temp", "vib", "lat"]
feature_min = sensor[feature_cols].min()
feature_range = (sensor[feature_cols].max() - feature_min).replace(0, 1)
glyph_rows = sensor.sort_values("risk_score", ascending=False).groupby("subsystem", observed=True).head(6).copy()
for _, row in glyph_rows.iterrows():
    vals = ((row[feature_cols] - feature_min) / feature_range).to_numpy(dtype=float)
    x0, y0 = float(row["x"]), float(row["y"])
    ax.add_patch(Circle((x0, y0), 0.18, fc="white", ec=group_palette[row["subsystem"]], lw=1.0, alpha=0.95))
    for k, val in enumerate(vals):
        bar_w = 0.055
        xpos = x0 - 0.13 + k * 0.07
        ax.add_patch(Rectangle((xpos, y0 - 0.13), bar_w, 0.24 * val, fc=[PALETTE["blue"], PALETTE["teal"], PALETTE["gold"], PALETTE["red"]][k], ec="none", alpha=0.85))
    ax.text(x0, y0 + 0.24, row["sensor_id"][-2:], ha="center", va="bottom", fontsize=6)
style_axis(ax, "Glyphs: four subchannels need space", equal=True, xlabel="floor x", ylabel="floor y")
ax.set_xlim(sensor["x"].min() - 0.4, sensor["x"].max() + 0.4)
ax.set_ylim(sensor["y"].min() - 0.4, sensor["y"].max() + 0.55)
legend_handles = [Rectangle((0, 0), 1, 1, fc=color) for color in [PALETTE["blue"], PALETTE["teal"], PALETTE["gold"], PALETTE["red"]]]
ax.legend(legend_handles, feature_names, loc="upper right", fontsize=7, frameon=True)

fig.suptitle("Marks and channels: the same data supports different reading tasks", x=0.02, ha="left", fontsize=13, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.96])
marks_path = save_matplotlib(fig, TOPIC, "marks-channels-encoding-ladder.png")
close(fig)

marks_checks = {
    "point_count": int(len(sensor)),
    "hue_category_count": int(len(groups)),
    "hue_category_count_at_most_eight": bool(len(groups) <= 8),
    "selected_count": int(selected.sum()),
    "area_cell_count_conserved": bool(int(counts.sum()) == len(sensor)),
    "glyph_count": int(len(glyph_rows)),
    "glyph_value_min": float(((glyph_rows[feature_cols] - feature_min) / feature_range).min().min()),
    "glyph_value_max": float(((glyph_rows[feature_cols] - feature_min) / feature_range).max().max()),
    "mark_families": ["point", "line", "area/cell", "glyph"],
}
marks_checks_path = save_json(marks_checks, TOPIC, "marks-channels-checks.json")
chapter_paths["figures"].append(marks_path)
chapter_paths["checks"].append(marks_checks_path)

display_artifact(marks_path, width=900)
display_artifact(marks_checks_path)


## 3. Color As A Channel: Ordered Values Need Ordered Lightness

Color is tempting because it is visually strong, but hue, lightness, and saturation do different jobs. Hue is useful for a small number of categories. Ordered or quantitative fields are safer when the colormap has monotone perceived lightness. Otherwise equal numerical steps can look unequal, and false boundaries can appear where the data are smooth.

The next artifact compares a rainbow-like map against a sequential map on the same scalar field. The check is deliberately mechanical: sample the colormap, convert RGB to relative luminance, and count reversals. A reversal means a larger data value is drawn with lower luminance than a smaller value somewhere along the ramp.


In [ ]:
def relative_luminance(rgb):
    rgb = np.asarray(rgb)[..., :3]
    return 0.2126 * rgb[..., 0] + 0.7152 * rgb[..., 1] + 0.0722 * rgb[..., 2]

grid_x = np.linspace(-3.4, 3.4, 220)
grid_y = np.linspace(-2.6, 2.6, 180)
GX, GY = np.meshgrid(grid_x, grid_y)
field = (
    0.70 * np.exp(-((GX - 1.25) ** 2 + (GY - 0.85) ** 2) / 1.35)
    + 0.42 * np.exp(-((GX + 1.9) ** 2 + (GY + 1.2) ** 2) / 0.9)
    + 0.13 * np.sin(2.4 * GX) * np.cos(1.9 * GY)
)
field = (field - field.min()) / (field.max() - field.min())
ramps = np.linspace(0, 1, 256)
jet_lum = relative_luminance(mpl.colormaps["jet"](ramps))
seq_cmap_name = "cividis"
seq_lum = relative_luminance(mpl.colormaps[seq_cmap_name](ramps))
jet_reversals = int(np.sum(np.diff(jet_lum) < -1e-4))
seq_reversals = int(np.sum(np.diff(seq_lum) < -1e-4))

fig, axes = plt.subplots(2, 2, figsize=(11, 7), gridspec_kw={"height_ratios": [4, 1.15]})
for ax, cmap_name, title in [(axes[0, 0], "jet", "Rainbow hue order"), (axes[0, 1], seq_cmap_name, "Sequential lightness order")]:
    im = ax.imshow(field, origin="lower", extent=[grid_x.min(), grid_x.max(), grid_y.min(), grid_y.max()], cmap=cmap_name)
    ax.contour(GX, GY, field, levels=[0.25, 0.5, 0.75], colors="white", linewidths=0.55, alpha=0.85)
    style_axis(ax, title, equal=True, xlabel="x", ylabel="y")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
axes[1, 0].plot(ramps, jet_lum, color=PALETTE["red"], lw=2)
axes[1, 0].set_title(f"Rainbow luminance reversals: {jet_reversals}", fontsize=10)
axes[1, 0].set_xlabel("normalized data value")
axes[1, 0].set_ylabel("relative luminance")
axes[1, 0].grid(True, color="#d7dde5", linewidth=0.7)
axes[1, 1].plot(ramps, seq_lum, color=PALETTE["blue"], lw=2)
axes[1, 1].set_title(f"{seq_cmap_name} luminance reversals: {seq_reversals}", fontsize=10)
axes[1, 1].set_xlabel("normalized data value")
axes[1, 1].set_ylabel("relative luminance")
axes[1, 1].grid(True, color="#d7dde5", linewidth=0.7)
fig.suptitle("Color validation: do ordered values also look ordered?", x=0.02, ha="left", fontsize=13, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.95])
color_path = save_matplotlib(fig, TOPIC, "color-channel-fit-validation.png")
close(fig)

color_checks = {
    "rainbow_colormap": "jet",
    "sequential_colormap": seq_cmap_name,
    "jet_luminance_reversals": jet_reversals,
    "sequential_luminance_reversals": seq_reversals,
    "jet_luminance_spearman": float(stats.spearmanr(ramps, jet_lum).correlation),
    "sequential_luminance_spearman": float(stats.spearmanr(ramps, seq_lum).correlation),
    "field_min": float(field.min()),
    "field_max": float(field.max()),
}
color_checks_path = save_json(color_checks, TOPIC, "color-channel-fit-checks.json")
chapter_paths["figures"].append(color_path)
chapter_paths["checks"].append(color_checks_path)

display_artifact(color_path, width=900)
display_artifact(color_checks_path)


## 4. Interaction: Overview First, Filter, Details On Demand

Interaction can be described as state changes on the visual encoding. A zoom changes the viewport. Sorting changes spatial order. Filtering changes the visible set. A hover or selection exposes details that would clutter the overview if they were always printed.

The Plotly artifact below is intentionally a small version of that pattern. The map is an overview, the red outline is the active high-risk selection predicate, the histogram shows how the selection sits within the full risk distribution, the hourly bars summarize where selected items occur, and the table supplies details on demand. Native Plotly controls provide pan, zoom, legend toggling, and hover details in the saved HTML.


In [ ]:
plot_df = sensor.copy()
plot_df["selected"] = selected
plot_df["size"] = 6 + 18 * plot_df["risk_score"]
plot_df["hover"] = (
    plot_df["sensor_id"]
    + "<br>subsystem=" + plot_df["subsystem"].astype(str)
    + "<br>risk=" + plot_df["risk_score"].round(3).astype(str)
    + "<br>uncertainty=" + plot_df["uncertainty"].round(3).astype(str)
    + "<br>status=" + plot_df["status"].astype(str)
)

detail_rows = plot_df.loc[selected].sort_values("risk_score", ascending=False).head(12)
selected_by_hour = plot_df.loc[selected].groupby("hour").size().reindex(hours, fill_value=0)
all_by_hour = plot_df.groupby("hour").size().reindex(hours, fill_value=0)

fig = make_subplots(
    rows=2,
    cols=2,
    specs=[[{"type": "xy"}, {"type": "xy"}], [{"type": "xy"}, {"type": "table"}]],
    subplot_titles=("Overview map with selected high-risk subset", "Risk distribution", "Selected items by hour", "Details on demand"),
    column_widths=[0.58, 0.42],
    row_heights=[0.62, 0.38],
    horizontal_spacing=0.09,
    vertical_spacing=0.14,
)
for group in groups:
    rows = plot_df[plot_df["subsystem"] == group]
    fig.add_trace(
        go.Scattergl(
            x=rows["x"], y=rows["y"], mode="markers", name=f"{group} all",
            marker={"size": rows["size"], "color": group_palette[group], "opacity": 0.36, "line": {"width": 0}},
            text=rows["hover"], hovertemplate="%{text}<extra></extra>", showlegend=True,
        ),
        row=1, col=1,
    )
sel = plot_df[selected]
fig.add_trace(
    go.Scattergl(
        x=sel["x"], y=sel["y"], mode="markers", name="selected predicate",
        marker={"size": 12 + 22 * sel["risk_score"], "color": "rgba(0,0,0,0)", "line": {"color": PALETTE["red"], "width": 2}},
        text=sel["hover"], hovertemplate="%{text}<extra></extra>", showlegend=True,
    ),
    row=1, col=1,
)
fig.add_trace(go.Histogram(x=plot_df["risk_score"], nbinsx=36, marker_color="#aab4c0", opacity=0.72, name="all risk"), row=1, col=2)
fig.add_trace(go.Histogram(x=sel["risk_score"], nbinsx=24, marker_color=PALETTE["red"], opacity=0.78, name="selected risk"), row=1, col=2)
fig.add_trace(go.Bar(x=hours, y=all_by_hour, marker_color="#d7dde5", name="all observations"), row=2, col=1)
fig.add_trace(go.Bar(x=hours, y=selected_by_hour, marker_color=PALETTE["red"], name="selected observations"), row=2, col=1)
fig.add_trace(
    go.Table(
        header={"values": ["sensor", "subsystem", "hour", "risk", "uncertainty", "status"], "fill_color": "#e5e7eb", "align": "left"},
        cells={
            "values": [
                detail_rows["sensor_id"], detail_rows["subsystem"], detail_rows["hour"],
                detail_rows["risk_score"].round(3), detail_rows["uncertainty"].round(3), detail_rows["status"].astype(str),
            ],
            "fill_color": "white",
            "align": "left",
        },
    ),
    row=2, col=2,
)
fig.update_xaxes(title_text="floor x", row=1, col=1)
fig.update_yaxes(title_text="floor y", scaleanchor="x", scaleratio=1, row=1, col=1)
fig.update_xaxes(title_text="risk score", row=1, col=2)
fig.update_yaxes(title_text="count", row=1, col=2)
fig.update_xaxes(title_text="hour", row=2, col=1)
fig.update_yaxes(title_text="observations", row=2, col=1)
fig.update_layout(
    title="Overview, filter, details: interaction as visible state",
    barmode="overlay",
    width=1050,
    height=720,
    legend={"orientation": "h", "y": -0.09},
    margin={"l": 35, "r": 25, "t": 80, "b": 80},
)
interaction_path = save_plotly_html(fig, TOPIC, "overview-filter-details-dashboard.html")
interaction_checks = {
    "total_count": int(len(plot_df)),
    "selected_count": int(selected.sum()),
    "selected_fraction": float(selected.mean()),
    "detail_rows": int(len(detail_rows)),
    "predicate": "risk_score >= 0.36 and uncertainty <= 72nd percentile",
    "every_selected_satisfies_risk_threshold": bool((plot_df.loc[selected, "risk_score"] >= 0.36).all()),
    "hour_bins_with_selected_items": int((selected_by_hour > 0).sum()),
}
interaction_checks_path = save_json(interaction_checks, TOPIC, "interaction-selection-checks.json")
chapter_paths["html"].append(interaction_path)
chapter_paths["checks"].append(interaction_checks_path)

display_artifact(interaction_path, width="100%", height=560)
display_artifact(interaction_checks_path)


## 5. Composition: Single Views, Layers, Small Multiples, And Linked Views

Composition decides where comparison happens. A single drawing makes shared-position comparison cheap but can become cluttered. Layering keeps focus items visible while background context remains faint. Small multiples trade screen space for clear side-by-side comparison. Linked views show the same selected items in different encodings, which is often more useful than adding yet another channel to one crowded display.

Relational data adds another composition choice: node-link views make paths and neighborhoods visible, while matrix views remove edge crossings and make dense communities easier to spot. Neither is universally best; the task decides.


In [ ]:
fig = plt.figure(figsize=(13, 7.8))
gs = fig.add_gridspec(2, 4, height_ratios=[1.05, 1.0], hspace=0.36, wspace=0.34)
ax_line = fig.add_subplot(gs[0, 0])
ax_layer = fig.add_subplot(gs[0, 1])
ax_hist = fig.add_subplot(gs[0, 2])
ax_text = fig.add_subplot(gs[0, 3])

for group in groups:
    rows = time_series[time_series["subsystem"] == group]
    ax_line.plot(rows["hour"], rows["mean_load_kw"], color=group_palette[group], lw=2, label=group)
style_axis(ax_line, "Single shared frame", xlabel="hour", ylabel="mean load")
ax_line.legend(fontsize=7, frameon=True)

ax_layer.scatter(sensor["x"], sensor["y"], s=12, c="#cbd5e1", alpha=0.42, edgecolors="none", label="context")
ax_layer.scatter(sensor.loc[selected, "x"], sensor.loc[selected, "y"], s=34 + 170 * sensor.loc[selected, "risk_score"], c=PALETTE["red"], alpha=0.82, edgecolors="white", linewidths=0.45, label="focus")
style_axis(ax_layer, "Layered focus and context", equal=True, xlabel="floor x", ylabel="floor y")
ax_layer.legend(fontsize=7, frameon=True)

bins = np.linspace(0, max(0.7, sensor["risk_score"].max()), 26)
ax_hist.hist(sensor["risk_score"], bins=bins, color="#cbd5e1", alpha=0.8, label="all")
ax_hist.hist(sensor.loc[selected, "risk_score"], bins=bins, color=PALETTE["red"], alpha=0.82, label="selected")
style_axis(ax_hist, "Linked risk summary", xlabel="risk", ylabel="count")
ax_hist.legend(fontsize=7, frameon=True)

ax_text.axis("off")
summary_lines = [
    "Same selection in all views",
    f"observations: {len(sensor)}",
    f"selected: {int(selected.sum())}",
    f"subsystems hit: {sensor.loc[selected, 'subsystem'].nunique()}",
    f"median uncertainty: {sensor.loc[selected, 'uncertainty'].median():.3f}",
]
for i, line in enumerate(summary_lines):
    ax_text.text(0.02, 0.92 - i * 0.16, line, ha="left", va="top", fontsize=10, color=PALETTE["ink"])
ax_text.set_title("Detail panel", loc="left", fontsize=11)

xlim = (sensor["x"].min() - 0.4, sensor["x"].max() + 0.4)
ylim = (sensor["y"].min() - 0.4, sensor["y"].max() + 0.4)
small_axes = [fig.add_subplot(gs[1, i]) for i in range(4)]
for ax, group in zip(small_axes, groups):
    rows = sensor[sensor["subsystem"] == group]
    ax.scatter(rows["x"], rows["y"], s=14 + 115 * rows["risk_score"], c=rows["risk_score"], cmap="cividis", vmin=0, vmax=0.65, alpha=0.82, edgecolors="white", linewidths=0.25)
    hit = rows.index.intersection(sensor[selected].index)
    ax.scatter(sensor.loc[hit, "x"], sensor.loc[hit, "y"], s=55, facecolors="none", edgecolors=PALETTE["red"], linewidths=1.0)
    style_axis(ax, f"Small multiple: {group}", equal=True, xlabel="x", ylabel="y")
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
fig.suptitle("Composition choices for the same selection state", x=0.02, ha="left", fontsize=13, fontweight="bold")
composition_path = save_matplotlib(fig, TOPIC, "composition-linked-views.png")
close(fig)

composition_checks = {
    "selected_count": int(selected.sum()),
    "selected_by_subsystem": {str(k): int(v) for k, v in sensor.loc[selected].groupby("subsystem", observed=True).size().items()},
    "small_multiple_shared_xlim": [float(xlim[0]), float(xlim[1])],
    "small_multiple_shared_ylim": [float(ylim[0]), float(ylim[1])],
    "views": ["single shared frame", "layered focus/context", "linked histogram", "detail panel", "small multiples"],
}
composition_checks_path = save_json(composition_checks, TOPIC, "composition-linked-view-checks.json")
chapter_paths["figures"].append(composition_path)
chapter_paths["checks"].append(composition_checks_path)

display_artifact(composition_path, width=900)
display_artifact(composition_checks_path)


In [ ]:
# Relational data example: the same graph as node-link and adjacency matrix.
community_sizes = [7, 8, 7]
rel_graph = nx.Graph()
node_community = {}
node_names = []
for c, size in enumerate(community_sizes):
    for i in range(size):
        node = f"C{c}-{i}"
        rel_graph.add_node(node, community=c)
        node_community[node] = c
        node_names.append(node)
# Ensure each community is connected, then add probabilistic within/between edges.
for c, size in enumerate(community_sizes):
    members = [n for n in node_names if node_community[n] == c]
    for a, b in zip(members, members[1:] + members[:1]):
        rel_graph.add_edge(a, b)
for i, u in enumerate(node_names):
    for v in node_names[i + 1:]:
        same = node_community[u] == node_community[v]
        threshold = 0.30 if same else 0.035
        if rng.random() < threshold:
            rel_graph.add_edge(u, v)
bridge_edges = [("C0-2", "C1-0"), ("C1-3", "C2-1"), ("C0-5", "C2-4")]
rel_graph.add_edges_from(bridge_edges)
path_nodes = nx.shortest_path(rel_graph, "C0-0", "C2-6")
path_edges = list(zip(path_nodes, path_nodes[1:]))
pos = nx.spring_layout(rel_graph, seed=23, k=0.85)

def orient(a, b, c):
    return (b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0])

def crossing_count(graph, positions):
    edges_list = list(graph.edges())
    count = 0
    for i, (a, b) in enumerate(edges_list):
        for c, d in edges_list[i + 1:]:
            if len({a, b, c, d}) < 4:
                continue
            p1, p2, p3, p4 = positions[a], positions[b], positions[c], positions[d]
            if (orient(p1, p2, p3) * orient(p1, p2, p4) < 0) and (orient(p3, p4, p1) * orient(p3, p4, p2) < 0):
                count += 1
    return count

ordered_nodes = sorted(node_names, key=lambda n: (node_community[n], n))
adj = nx.to_numpy_array(rel_graph, nodelist=ordered_nodes, dtype=int)
node_to_order = {n: i for i, n in enumerate(ordered_nodes)}
node_colors = [[PALETTE["teal"], PALETTE["blue"], PALETTE["gold"]][node_community[n]] for n in rel_graph.nodes()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.8))
nx.draw_networkx_edges(rel_graph, pos, ax=ax1, edge_color="#9aa4b2", width=1.0, alpha=0.65)
nx.draw_networkx_edges(rel_graph, pos, edgelist=path_edges, ax=ax1, edge_color=PALETTE["red"], width=2.5)
nx.draw_networkx_nodes(rel_graph, pos, ax=ax1, node_color=node_colors, node_size=250, edgecolors="white", linewidths=0.8)
nx.draw_networkx_labels(rel_graph, pos, ax=ax1, font_size=6)
ax1.set_title("Node-link view: paths are direct, crossings compete")
ax1.axis("off")

ax2.imshow(adj, cmap="Greys", interpolation="nearest", vmin=0, vmax=1)
for edge in path_edges:
    i, j = node_to_order[edge[0]], node_to_order[edge[1]]
    ax2.scatter([j, i], [i, j], s=70, facecolors="none", edgecolors=PALETTE["red"], linewidths=1.4)
start = 0
for size in community_sizes[:-1]:
    start += size
    ax2.axhline(start - 0.5, color=PALETTE["blue"], lw=1.0)
    ax2.axvline(start - 0.5, color=PALETTE["blue"], lw=1.0)
ax2.set_xticks(range(len(ordered_nodes)))
ax2.set_yticks(range(len(ordered_nodes)))
ax2.set_xticklabels(ordered_nodes, rotation=90, fontsize=6)
ax2.set_yticklabels(ordered_nodes, fontsize=6)
ax2.set_title("Matrix view: communities are blocks, crossings disappear")
ax2.set_xlabel("node")
ax2.set_ylabel("node")
fig.suptitle("Relational composition: node-link and matrix views answer different graph tasks", x=0.02, ha="left", fontsize=13, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.93])
graph_matrix_path = save_matplotlib(fig, TOPIC, "graph-matrix-composition-comparison.png")
close(fig)

matrix_edge_count = int(adj.sum() // 2)
graph_checks = {
    "node_count": rel_graph.number_of_nodes(),
    "edge_count": rel_graph.number_of_edges(),
    "matrix_edge_count": matrix_edge_count,
    "matrix_matches_graph_edge_count": bool(matrix_edge_count == rel_graph.number_of_edges()),
    "node_link_crossing_count_in_this_layout": crossing_count(rel_graph, pos),
    "highlighted_shortest_path_length": len(path_edges),
    "communities": community_sizes,
}
graph_checks_path = save_json(graph_checks, TOPIC, "graph-matrix-composition-checks.json")
chapter_paths["figures"].append(graph_matrix_path)
chapter_paths["checks"].append(graph_checks_path)

display_artifact(graph_matrix_path, width=900)
display_artifact(graph_checks_path)


## 6. Data Reduction: Aggregate, Filter, Focus, And Project

Pixels, time, and attention are limited resources. Reduction is not a failure; it is an explicit design move. Aggregation replaces many items with one summary mark. Filtering hides items that do not match the current task. Focus+context changes the geometry of the view so that a focus region receives more space while context remains visible. Dimensionality reduction projects many measured dimensions down to a smaller display space.

Every reduction below records what was removed or preserved. The checks test count conservation for aggregation, subset size for filtering, radial monotonicity for the focus+context lens, and orthonormality for PCA.


In [ ]:
feature_cols = ["load_kw", "temperature_c", "vibration", "latency_ms", "risk_score", "uncertainty"]
X = sensor[feature_cols].to_numpy(dtype=float)
X_mean = X.mean(axis=0)
X_std = X.std(axis=0, ddof=1)
Xz = (X - X_mean) / X_std
U, S, Vt = np.linalg.svd(Xz, full_matrices=False)
pca_scores = U[:, :2] * S[:2]
explained_ratio = (S ** 2) / np.sum(S ** 2)
orth_error = float(np.max(np.abs(Vt[:2] @ Vt[:2].T - np.eye(2))))

counts_reduction, rx, ry = np.histogram2d(sensor["x"], sensor["y"], bins=32)
risk_reduction_sum, _, _ = np.histogram2d(sensor["x"], sensor["y"], bins=[rx, ry], weights=sensor["risk_score"])
mean_risk_reduction = np.divide(risk_reduction_sum, counts_reduction, out=np.full_like(risk_reduction_sum, np.nan, dtype=float), where=counts_reduction > 0)

fig, axes = plt.subplots(2, 2, figsize=(12, 8.2))
ax = axes[0, 0]
ax.scatter(sensor["x"], sensor["y"], s=8, c="#8fa0b3", alpha=0.34, edgecolors="none")
style_axis(ax, "Raw items: accurate but cluttered", equal=True, xlabel="floor x", ylabel="floor y")

ax = axes[0, 1]
im = ax.imshow(mean_risk_reduction.T, origin="lower", extent=[rx[0], rx[-1], ry[0], ry[-1]], cmap="cividis", vmin=0, vmax=max(0.55, float(np.nanmax(mean_risk_reduction))))
style_axis(ax, "Aggregation: one cell summarizes many items", equal=True, xlabel="floor x", ylabel="floor y")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03, label="mean risk")

ax = axes[1, 0]
ax.scatter(sensor["x"], sensor["y"], s=9, c="#d1d5db", alpha=0.35, edgecolors="none")
ax.scatter(sensor.loc[selected, "x"], sensor.loc[selected, "y"], s=34, c=PALETTE["red"], alpha=0.85, label="risk filter")
ax.scatter(sensor.loc[high_uncertainty, "x"], sensor.loc[high_uncertainty, "y"], s=54, facecolors="none", edgecolors=PALETTE["blue"], linewidths=0.8, label="uncertainty filter")
style_axis(ax, "Filtering: fewer items, clearer task", equal=True, xlabel="floor x", ylabel="floor y")
ax.legend(loc="upper right", fontsize=7, frameon=True)

ax = axes[1, 1]
for group in groups:
    rows = sensor["subsystem"] == group
    ax.scatter(pca_scores[rows, 0], pca_scores[rows, 1], s=13, color=group_palette[group], alpha=0.55, label=group, edgecolors="none")
ax.scatter(pca_scores[selected, 0], pca_scores[selected, 1], s=45, facecolors="none", edgecolors=PALETTE["red"], linewidths=0.9)
style_axis(ax, f"PCA projection: PC1 {explained_ratio[0]:.0%}, PC2 {explained_ratio[1]:.0%}", xlabel="PC1 score", ylabel="PC2 score")
ax.legend(loc="upper right", fontsize=7, ncol=2, frameon=True)

fig.suptitle("Reduction workflow: preserve the task while reducing what must be drawn", x=0.02, ha="left", fontsize=13, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.95])
reduction_path = save_matplotlib(fig, TOPIC, "reduction-aggregation-pca-workflow.png")
close(fig)

coords = sensor[["x", "y"]].to_numpy(dtype=float)
focus = coords[selected].mean(axis=0)
deltas = coords - focus
r = np.linalg.norm(deltas, axis=1)
rmax = float(r.max())
unit = np.divide(deltas, r[:, None], out=np.zeros_like(deltas), where=r[:, None] > 1e-12)
distortion = 3.5
scaled = r / rmax
r_new = ((distortion + 1) * scaled / (distortion * scaled + 1)) * rmax
fisheye_coords = focus + unit * r_new[:, None]
order = np.argsort(r)
radial_monotone = bool(np.all(np.diff(r_new[order]) >= -1e-9))
boundary_error = float(abs(r_new[np.argmax(r)] - rmax))

fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
for ax, pts, title in [(axes[0], coords, "Original layout"), (axes[1], fisheye_coords, "Focus+context fisheye")]:
    ax.scatter(pts[:, 0], pts[:, 1], s=9, c="#a7b4c4", alpha=0.42, edgecolors="none")
    ax.scatter(pts[selected, 0], pts[selected, 1], s=34, c=PALETTE["red"], alpha=0.82, edgecolors="white", linewidths=0.35)
    ax.scatter([focus[0]], [focus[1]], s=80, c=PALETTE["ink"], marker="x", linewidths=2)
    circle = Circle(focus, radius=np.quantile(r, 0.18), fill=False, ec=PALETTE["red"], lw=1.2, ls="--", alpha=0.9)
    ax.add_patch(circle)
    style_axis(ax, title, equal=True, xlabel="floor x", ylabel="floor y")
fig.suptitle("Focus+context: local detail expands while global context remains", x=0.02, ha="left", fontsize=13, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.92])
focus_context_path = save_matplotlib(fig, TOPIC, "focus-context-fisheye-lens.png")
close(fig)

reduction_checks = {
    "aggregation_total_count": int(counts_reduction.sum()),
    "aggregation_count_error": int(abs(counts_reduction.sum() - len(sensor))),
    "selected_filter_count": int(selected.sum()),
    "high_uncertainty_filter_count": int(high_uncertainty.sum()),
    "pca_explained_ratio_first_two": [float(explained_ratio[0]), float(explained_ratio[1])],
    "pca_orthonormal_error": orth_error,
    "fisheye_focus": [float(focus[0]), float(focus[1])],
    "fisheye_radial_monotone": radial_monotone,
    "fisheye_boundary_error": boundary_error,
}
reduction_checks_path = save_json(reduction_checks, TOPIC, "reduction-workflow-checks.json")
chapter_paths["figures"].extend([reduction_path, focus_context_path])
chapter_paths["checks"].append(reduction_checks_path)

display_artifact(reduction_path, width=900)
display_artifact(focus_context_path, width=900)
display_artifact(reduction_checks_path)


## 7. Validation And Uncertainty

Validation happens at several levels. The domain question may be wrong, the data abstraction may omit the important variable, the visual encoding may violate perceptual principles, or the algorithm may be too slow or incorrect. This notebook can only test the lower layers directly, but it still records the higher-layer assumptions so they can be challenged.

Uncertainty deserves a visible encoding because a confident medium-risk point and an uncertain high-risk point call for different decisions. The artifact below separates predicted risk from uncertainty intervals and tests calibration against the simulated anomaly outcomes. In real work, the outcome labels would come from field verification or user studies rather than the synthetic generator.


In [ ]:
uncertainty_scale = 1.96
lower = np.clip(sensor["risk_score"] - uncertainty_scale * sensor["uncertainty"], 0, 1)
upper = np.clip(sensor["risk_score"] + uncertainty_scale * sensor["uncertainty"], 0, 1)
interval_coverage = float(((sensor["true_risk"] >= lower) & (sensor["true_risk"] <= upper)).mean())

calib = sensor.assign(bin=pd.cut(sensor["risk_score"], bins=np.linspace(0, 1, 11), include_lowest=True)).groupby("bin", observed=True).agg(
    n=("risk_score", "size"),
    predicted=("risk_score", "mean"),
    observed=("anomaly", "mean"),
).reset_index(drop=True)
calib = calib[calib["n"] > 0].copy()
calib["stderr"] = np.sqrt(np.maximum(calib["observed"] * (1 - calib["observed"]), 1e-6) / calib["n"])
calibration_mae = float(np.average(np.abs(calib["observed"] - calib["predicted"]), weights=calib["n"]))

top = sensor.assign(lower=lower, upper=upper).sort_values("risk_score", ascending=False).head(22).copy()
top = top.iloc[::-1]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1.05, 1.0, 1.25]})
ax = axes[0]
unc_sizes = 18 + 420 * sensor["uncertainty"]
sc = ax.scatter(sensor["x"], sensor["y"], s=unc_sizes, c=sensor["risk_score"], cmap="cividis", vmin=0, vmax=0.7, alpha=0.52, edgecolors="none")
ax.scatter(sensor.loc[high_uncertainty, "x"], sensor.loc[high_uncertainty, "y"], s=80, facecolors="none", edgecolors=PALETTE["blue"], linewidths=0.8, label="high uncertainty")
style_axis(ax, "Risk color, uncertainty size", equal=True, xlabel="floor x", ylabel="floor y")
fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.03, label="risk")
ax.legend(loc="upper right", fontsize=7, frameon=True)

ax = axes[1]
ax.errorbar(calib["predicted"], calib["observed"], yerr=1.96 * calib["stderr"], fmt="o", color=PALETTE["blue"], ecolor="#8fb3d9", capsize=3)
ax.plot([0, 1], [0, 1], color="#6b7280", lw=1.0, ls="--", label="perfect calibration")
for _, row in calib.iterrows():
    ax.text(row["predicted"], row["observed"] + 0.035, str(int(row["n"])), ha="center", va="bottom", fontsize=7, color=PALETTE["ink"])
style_axis(ax, f"Calibration MAE {calibration_mae:.3f}", xlabel="mean predicted risk", ylabel="observed anomaly rate")
ax.set_xlim(0, max(0.72, float(calib["predicted"].max()) + 0.05))
ax.set_ylim(0, max(0.72, float(calib["observed"].max()) + 0.12))
ax.legend(fontsize=7, frameon=True)

ax = axes[2]
ypos = np.arange(len(top))
xerr = np.vstack([(top["risk_score"] - top["lower"]).to_numpy(), (top["upper"] - top["risk_score"]).to_numpy()])
colors = np.where(top["anomaly"], PALETTE["red"], PALETTE["gray"])
ax.errorbar(top["risk_score"], ypos, xerr=xerr, fmt="none", ecolor="#aab4c0", elinewidth=1.2, capsize=2)
ax.scatter(top["risk_score"], ypos, c=colors, s=28, zorder=3)
ax.set_yticks(ypos)
ax.set_yticklabels(top["sensor_id"], fontsize=7)
style_axis(ax, f"Top-risk intervals, coverage {interval_coverage:.1%}", xlabel="risk interval", ylabel="sensor")
ax.set_xlim(0, 1)

fig.suptitle("Validation workbench: show uncertainty and test the risk model", x=0.02, ha="left", fontsize=13, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.92])
uncertainty_path = save_matplotlib(fig, TOPIC, "uncertainty-validation-workbench.png")
close(fig)

uncertainty_checks = {
    "interval_scale": uncertainty_scale,
    "interval_coverage": interval_coverage,
    "calibration_mae": calibration_mae,
    "calibration_bins": int(len(calib)),
    "high_uncertainty_count": int(high_uncertainty.sum()),
    "top_interval_rows": int(len(top)),
}
uncertainty_checks_path = save_json(uncertainty_checks, TOPIC, "uncertainty-validation-checks.json")
chapter_paths["figures"].append(uncertainty_path)
chapter_paths["checks"].append(uncertainty_checks_path)

display_artifact(uncertainty_path, width=950)
display_artifact(uncertainty_checks_path)


## 8. Applied lab: A Design Review Ledger

A practical visualization workflow needs a review ledger that is more specific than "make a chart." Each row below states a domain question, the abstract task, the data type involved, the proposed visual encoding, the interaction state, the reduction strategy, and the validation check. This is the notebook's compact design contract. To explore, change a row, rerun the downstream visualization, and see which checks no longer match.


In [ ]:
review_rows = [
    {
        "domain_question": "Which facility areas need inspection now?",
        "data_type": "spatial table with quantitative risk",
        "abstract_task": "find outliers and locate them",
        "marks_channels": "point marks; x/y position, categorical hue, risk size",
        "interaction": "filter by risk and uncertainty; hover for details",
        "reduction": "aggregate to cell mean risk when clutter rises",
        "validation": "selection count, nonblank artifact, task reaches validation node",
    },
    {
        "domain_question": "Which subsystem causes the daily load peak?",
        "data_type": "ordered temporal measurements by category",
        "abstract_task": "compare trends and extrema",
        "marks_channels": "line marks; time on x, load on y, subsystem hue",
        "interaction": "legend toggle and zoom",
        "reduction": "cluster or average curves before drawing many series",
        "validation": "shared axes and preserved group labels",
    },
    {
        "domain_question": "How are suspected failures connected?",
        "data_type": "relational graph with communities",
        "abstract_task": "follow paths and identify communities",
        "marks_channels": "node-link plus adjacency matrix",
        "interaction": "highlight a path or selected community",
        "reduction": "community ordering or aggregation",
        "validation": "matrix edge count equals graph edge count",
    },
    {
        "domain_question": "Which warnings are trustworthy enough to act on?",
        "data_type": "probabilistic score with uncertainty and outcome labels",
        "abstract_task": "judge confidence and validate calibration",
        "marks_channels": "interval marks; risk position, uncertainty width/size",
        "interaction": "sort by risk and inspect details",
        "reduction": "bin predictions for calibration",
        "validation": "interval coverage and calibration error",
    },
]
ledger_path = save_table_csv(review_rows, TOPIC, "design-review-ledger.csv")
chapter_paths["tables"].append(ledger_path)
review_ledger = pd.DataFrame(review_rows)
display(review_ledger)
display_artifact(ledger_path)


## 9. Sanity checks

The checks combine artifact integrity with chapter-specific invariants. They are intentionally narrower than a full user study, but they prevent common notebook failures: missing files, blank images, stale paths, decorative figures without data, broken reductions, and uncertainty claims without numerical backing.


In [ ]:
all_paths_before_final = [*chapter_paths["figures"], *chapter_paths["html"], *chapter_paths["checks"], *chapter_paths["tables"]]
artifact_records = assert_artifacts(all_paths_before_final)
image_records = [assert_nonblank_image(path) for path in chapter_paths["figures"]]

assert data_task_checks["is_dag"] and data_task_checks["all_tasks_reach_validation"]
assert marks_checks["hue_category_count_at_most_eight"]
assert 0 <= marks_checks["glyph_value_min"] <= marks_checks["glyph_value_max"] <= 1
assert color_checks["jet_luminance_reversals"] > color_checks["sequential_luminance_reversals"]
assert color_checks["sequential_luminance_reversals"] == 0
assert 0 < interaction_checks["selected_count"] < interaction_checks["total_count"]
assert graph_checks["matrix_matches_graph_edge_count"]
assert reduction_checks["aggregation_count_error"] == 0
assert reduction_checks["pca_orthonormal_error"] < 1e-12
assert reduction_checks["fisheye_radial_monotone"] and reduction_checks["fisheye_boundary_error"] < 1e-9
assert uncertainty_checks["interval_coverage"] >= 0.90
assert uncertainty_checks["calibration_mae"] <= 0.16

final_sanity = {
    "chapter": 23,
    "title": "Visualization",
    "source_span": "printed pp. 645-680 / PDF pp. 662-697",
    "artifact_count_before_final": len(all_paths_before_final),
    "png_count": len(chapter_paths["figures"]),
    "html_count": len(chapter_paths["html"]),
    "check_count_before_final": len(chapter_paths["checks"]),
    "table_count": len(chapter_paths["tables"]),
    "nonblank_images": len(image_records),
    "core_invariants": {
        "data_task_dag": data_task_checks["is_dag"],
        "all_tasks_reach_validation": data_task_checks["all_tasks_reach_validation"],
        "sequential_luminance_reversals": color_checks["sequential_luminance_reversals"],
        "selected_count": interaction_checks["selected_count"],
        "matrix_edges_match_graph": graph_checks["matrix_matches_graph_edge_count"],
        "aggregation_count_error": reduction_checks["aggregation_count_error"],
        "pca_orthonormal_error": reduction_checks["pca_orthonormal_error"],
        "fisheye_radial_monotone": reduction_checks["fisheye_radial_monotone"],
        "interval_coverage": uncertainty_checks["interval_coverage"],
        "calibration_mae": uncertainty_checks["calibration_mae"],
    },
    "artifacts": artifact_records,
}
final_sanity_path = save_json(final_sanity, TOPIC, "visualization-sanity-checks.json")
chapter_paths["checks"].append(final_sanity_path)
assert_artifacts([final_sanity_path])
display_artifact(final_sanity_path)
print("Chapter 23 sanity checks passed.")


## Takeaways

Visualization design is a chain of abstractions, not a last-mile plotting step. The most important move is to name the data type and task before choosing visual encodings.

Use spatial position for the quantities that require the most accurate reading. Use hue for a small number of categories, and use monotone lightness for ordered or quantitative fields. Treat size, opacity, glyph parts, and animation as lower-capacity channels that need justification.

Interaction is useful when it changes the state of the question: zoom changes the viewport, filtering changes the visible set, sorting changes spatial order, and details on demand reveal labels or measurements only when needed.

Composition and reduction are responses to limits. Single views support direct comparison but can clutter; linked views and small multiples spend more space to lower memory load. Aggregation, filtering, focus+context, and dimensionality reduction must state what they preserve and what they discard.

Validation belongs in the design, not after it. A complete visualization workflow checks perceptual fit, artifact integrity, algorithmic invariants, selection semantics, and uncertainty claims.
